# Insights Keywords
> Find missing queries and green keyword opportunities.

In [ ]:
# | default_exp insights.keywords

In [ ]:
#| export
from sqlmodel import Session
from seo_rat.gsc.queries import get_top_queries
from seo_rat.insights.trends import detect_query_trends


In [ ]:
# | export
STOP_WORDS = {
    "a",
    "an",
    "the",
    "is",
    "it",
    "in",
    "to",
    "for",
    "of",
    "and",
    "or",
    "how",
    "what",
    "who",
    "which",
    "this",
    "that",
    "هل",
    "في",
    "من",
    "على",
    "هو",
    "هذا",
    "ما",
}

def query_words_in_content(query: str,
                           content: str
                           ) -> bool:
    "Check if all significant words from a query appear in the content."
    words = [w for w in query.lower().split() if w not in STOP_WORDS and len(w) > 1]
    return all(w in content.lower() for w in words)


In [ ]:
# | export
from logging import Filter
import dspy


def all_missing_queries(queries: list[dict], content: str) -> list[dict]:
    "Find GSC queries whose significant words don't appear in the page content."
    return [q for q in queries if not query_words_in_content(q["query"], content)]


class FilterMissingQueries(dspy.Signature):
    """Filter a list of search queries to only those relevant to the article topic."""

    queries: list[str] = dspy.InputField(
        desc="List of missing search queries to evaluate"
    )
    focus_keyword: str = dspy.InputField(desc="Main topic of the article")
    secondary_keywords: list[str] = dspy.InputField(desc="Related subtopics")
    content_snippet: str = dspy.InputField(desc="First 500 chars of article content")

    relevant_queries: list[str] = dspy.OutputField(
        desc="Only the queries relevant to this article's topic"
    )


filter_missing = dspy.Predict(FilterMissingQueries)


def filter_missing_queries(
    queries: list[str],
    focus_keyword: str,
    secondary_keywords: list,
    content: str,
) -> list[str]:
    """Filter missing queries in one batched DSPy call."""
    if not queries:
        return []
    result = filter_missing(
        queries=queries,
        focus_keyword=focus_keyword,
        secondary_keywords=secondary_keywords,
        content_snippet=content[:500],
    )
    return result.relevant_queries



In [ ]:
# | export
def find_green_keywords(
    session: Session,
    site_url: str,
    page_path: str,
    content: str,
    days: int = 30,
    limit: int = 100,
) -> list[dict]:
    "Find emerging queries not yet covered in page content."
    trends = detect_query_trends(
        session, site_url, page_path=page_path, days=days, limit=limit
    )
    rising = [
        t for t in trends if t["trend"] == "rising" and t["prev_impressions"] <= 5
    ]
    return all_missing_queries(rising, content)
